# Notebook 3: Memoria y Persistencia con LangGraph

Hasta ahora nuestros grafos son **sin estado entre ejecuciones**: cada `invoke()` empieza de cero.
En este notebook aprenderás a construir grafos que **recuerdan** conversaciones anteriores.

Aprenderás:
- Cómo usar `MemorySaver` para persistir el estado del grafo
- Qué son los **checkpoints** y cómo LangGraph los gestiona
- El concepto de `thread_id` para separar conversaciones
- Cómo **retomar** una conversación desde un punto anterior
- Cómo construir un chatbot con historial real de mensajes

## 1. Instalación y configuración

In [ ]:
# %pip install -qU langgraph langchain-google-genai langchain-core

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7,
)
print("Modelo listo:", llm.model)

## 2. El problema: sin memoria, el chatbot olvida todo

Primero, veamos por qué necesitamos persistencia.

In [ ]:
# Sin memoria: cada llamada es independiente
from langchain_core.messages import HumanMessage, AIMessage

r1 = llm.invoke("Hola, mi nombre es Carlos")
print("Turno 1:", r1.content)

r2 = llm.invoke("¿Cómo me llamo?")
print("Turno 2:", r2.content)  # ← No sabe que se llama Carlos

## 3. La solución: estado con lista de mensajes + reducers

Para el historial de chat usamos un campo especial en el estado: una lista de mensajes.
LangGraph tiene un mecanismo especial llamado **reducer** que controla cómo se actualiza cada campo.

Con `Annotated[list, add_messages]`, los nuevos mensajes se **añaden** a la lista en lugar de reemplazarla.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class EstadoChat(TypedDict):
    # 'add_messages' es el reducer: AÑADE nuevos mensajes a la lista
    # en lugar de reemplazar toda la lista.
    # Sin reducer: estado['mensajes'] = nuevos_mensajes  (reemplaza)
    # Con reducer:  estado['mensajes'] += nuevos_mensajes (acumula)
    mensajes: Annotated[list[BaseMessage], add_messages]

print("Estado con reducer 'add_messages' definido")
print("Tipo de mensajes:", EstadoChat.__annotations__["mensajes"])

## 4. Nodo del chatbot

In [ ]:
from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = """Eres un asistente amigable que recuerda todo lo que el usuario te ha dicho
en la conversación. Personaliza tus respuestas con la información que el usuario ha compartido."""

def nodo_chatbot(estado: EstadoChat) -> dict:
    """
    Recibe el historial completo de mensajes y genera una respuesta.
    El historial incluye TODOS los turnos anteriores gracias al reducer.
    """
    # Preparar mensajes con sistema prompt
    mensajes_completos = [SystemMessage(content=SYSTEM_PROMPT)] + estado["mensajes"]
    
    respuesta = llm.invoke(mensajes_completos)
    
    # Retornamos solo el nuevo mensaje; el reducer lo añadirá al historial
    return {"mensajes": [respuesta]}

print("Nodo chatbot definido")

## 5. Construir el grafo con MemorySaver

`MemorySaver` guarda un **checkpoint** del estado después de cada nodo.
Los checkpoints se identifican con un `thread_id`, lo que permite múltiples conversaciones independientes.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Construir el grafo
grafo = StateGraph(EstadoChat)
grafo.add_node("chatbot", nodo_chatbot)
grafo.add_edge(START, "chatbot")
grafo.add_edge("chatbot", END)

# MemorySaver: guarda checkpoints en memoria (RAM)
# Para producción: usar SqliteSaver o PostgresSaver
memoria = MemorySaver()

# Compilar CON el checkpointer
app = grafo.compile(checkpointer=memoria)

print("✅ Chatbot con memoria compilado")

## 6. El thread_id: clave para múltiples conversaciones

Cada conversación tiene un `thread_id` único. LangGraph usa esto para:
- Guardar el estado asociado a esa conversación
- Recuperarlo en la siguiente llamada
- Mantener conversaciones completamente separadas

In [ ]:
from langchain_core.messages import HumanMessage

def chatear(app, mensaje: str, thread_id: str):
    """Envía un mensaje al chatbot dentro de un thread específico."""
    # El config es OBLIGATORIO cuando usamos checkpointer
    config = {"configurable": {"thread_id": thread_id}}
    
    estado_entrada = {"mensajes": [HumanMessage(content=mensaje)]}
    
    resultado = app.invoke(estado_entrada, config=config)
    
    # El último mensaje es la respuesta del AI
    respuesta = resultado["mensajes"][-1].content
    return respuesta

print("Función de chat lista")

In [ ]:
# Conversación 1: Usuario 'Carlos'
print("=" * 60)
print("CONVERSACIÓN DE CARLOS (thread: carlos-001)")
print("=" * 60)

r = chatear(app, "Hola, me llamo Carlos y soy programador de Python.", "carlos-001")
print(f"Usuario: Hola, me llamo Carlos y soy programador de Python.")
print(f"Bot: {r}\n")

r = chatear(app, "¿Cuál es mi lenguaje de programación favorito?", "carlos-001")
print(f"Usuario: ¿Cuál es mi lenguaje de programación favorito?")
print(f"Bot: {r}\n")

r = chatear(app, "Tengo 5 años de experiencia. ¿Me darías un consejo para mejorar?", "carlos-001")
print(f"Usuario: Tengo 5 años de experiencia. ¿Me darías un consejo para mejorar?")
print(f"Bot: {r}\n")

In [ ]:
# Conversación 2: Usuario 'Ana' (completamente independiente)
print("=" * 60)
print("CONVERSACIÓN DE ANA (thread: ana-001)")
print("=" * 60)

r = chatear(app, "Hola, soy Ana y trabajo en diseño gráfico.", "ana-001")
print(f"Usuario: Hola, soy Ana y trabajo en diseño gráfico.")
print(f"Bot: {r}\n")

r = chatear(app, "¿A qué me dedico?", "ana-001")
print(f"Usuario: ¿A qué me dedico?")
print(f"Bot: {r}\n")

In [ ]:
# Volvemos a Carlos: ¿recuerda quién es?
print("=" * 60)
print("DE VUELTA CON CARLOS (thread: carlos-001)")
print("=" * 60)

r = chatear(app, "¿Cómo me llamo y qué hago?", "carlos-001")
print(f"Usuario: ¿Cómo me llamo y qué hago?")
print(f"Bot: {r}")

## 7. Inspeccionar el estado guardado

LangGraph nos permite ver el estado actual de cualquier thread.

In [ ]:
# Ver el estado actual del thread de Carlos
config_carlos = {"configurable": {"thread_id": "carlos-001"}}
snapshot = app.get_state(config_carlos)

print("Estado actual del thread 'carlos-001':")
print(f"Número de mensajes en historial: {len(snapshot.values['mensajes'])}")
print("\nHistorial completo:")
for i, msg in enumerate(snapshot.values["mensajes"]):
    tipo = "Usuario" if msg.__class__.__name__ == "HumanMessage" else "Bot"
    print(f"  [{i+1}] {tipo}: {msg.content[:80]}..." if len(msg.content) > 80 else f"  [{i+1}] {tipo}: {msg.content}")

## 8. Ver el historial de checkpoints

In [ ]:
# Ver todos los checkpoints guardados para un thread
historial = list(app.get_state_history(config_carlos))

print(f"Total de checkpoints para 'carlos-001': {len(historial)}")
print("\nCheckpoints (del más reciente al más antiguo):")
for i, checkpoint in enumerate(historial):
    n_msgs = len(checkpoint.values.get("mensajes", []))
    ts = checkpoint.metadata.get("step", "?")
    print(f"  Checkpoint {i+1}: step={ts}, mensajes={n_msgs}")

## 9. Bonus: SqliteSaver para persistencia real

`MemorySaver` pierde los datos cuando termina el proceso Python.
Para persistencia real entre sesiones, usa `SqliteSaver`.

In [ ]:
%pip install -qU langgraph-checkpoint-sqlite

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

# Los datos se guardan en un archivo .db local
with SqliteSaver.from_conn_string("chatbot_historia.db") as db_memory:
    app_sqlite = grafo.compile(checkpointer=db_memory)
    
    config = {"configurable": {"thread_id": "persistente-001"}}
    
    # Esta conversación sobrevive entre reinicios del proceso
    r = app_sqlite.invoke(
        {"mensajes": [HumanMessage(content="Hola, soy una conversación persistente.")]},
        config=config
    )
    print("Bot:", r["mensajes"][-1].content)

print("\n✅ Conversación guardada en chatbot_historia.db")
print("Si ejecutas este notebook de nuevo, la conversación continuará desde aquí.")

## 10. Ejercicios propuestos

1. **Límite de memoria**: Modifica el nodo chatbot para que solo envíe los últimos 6 mensajes al LLM (ventana deslizante), aunque el historial completo se guarde en el estado.

2. **Resumen de conversación**: Añade un segundo nodo que se active cada 10 mensajes y resuma la conversación, almacenando el resumen en un campo `resumen` del estado.

3. **Múltiples usuarios**: Crea una función que genere `thread_id` únicos por usuario usando UUID, y simula 3 usuarios chateando en paralelo.

## Resumen

| Concepto | Descripción |
|---|---|
| `MemorySaver` | Guarda checkpoints en RAM (en proceso) |
| `SqliteSaver` | Guarda checkpoints en SQLite (entre sesiones) |
| `thread_id` | Identifica una conversación única |
| `add_messages` | Reducer que acumula mensajes en lugar de reemplazarlos |
| `get_state()` | Lee el estado actual de un thread |
| `get_state_history()` | Lista todos los checkpoints de un thread |
| `config` | Diccionario con `{"configurable": {"thread_id": ...}}` requerido para checkpointing |